# ⚽ Asking the Rulebook — end-to-end walkthrough

A retrieval-augmented (RAG) chatbot over the **IFAB/FIFA Laws of the Game 2025/26**.

> **Educational project — not affiliated with or endorsed by FIFA or IFAB.**
> Answers can be incomplete or wrong; verify against the official document.

This notebook walks through the *current* pipeline: corpus provenance → ingestion →
retrieval (`rag/`) → query expansion → conversation-aware retrieval → page-diverse
retrieval → evaluation (**Page Hit Rate@k**, a development set) → answer-level
framework → an optional generation demo (skips without a Groq key) → limitations.

All retrieval here uses the same `rag/` package as the production API, so the
notebook cannot drift from the deployed behaviour.

## 1. Setup

In [1]:
import os, sys, json
from pathlib import Path

# Robust repo-root resolution (no machine-specific absolute paths).
ROOT = Path.cwd()
if not (ROOT / "rag").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from rag import (Retriever, DEFAULT_CONFIG, expand_query, matched_phrases,
                 build_retrieval_query, extract_citations, validate_citations)
print("repo root:", ROOT.name)

repo root: fifa-rag-chatbot


## 2. Corpus provenance

The corpus is the official *Laws of the Game 2025/26* (single-pages edition),
published by IFAB/FIFA. We did **not** author it and have not obtained
reproduction permission; `data/chunks.json` is a derived research artifact. The
source PDF is **not** committed — provenance and checksums live in
`data/source_metadata.json`.

In [2]:
meta = json.loads((ROOT / "data" / "source_metadata.json").read_text())
for k in ["document_title","publisher","official_source_url","access_date",
          "source_pdf_page_count","chunk_count","indexed_page_range"]:
    print(f"{k:>22}: {meta[k]}")
print("\ncopyright:", meta["copyright_notice"][:160], "...")

        document_title: Laws of the Game 2025/26
             publisher: The International Football Association Board (IFAB)
   official_source_url: https://downloads.theifab.com/downloads/laws-of-the-game-2025-26-single-pages?l=en
           access_date: 2026-06-13
 source_pdf_page_count: 230
           chunk_count: 267
    indexed_page_range: [4, 217]

copyright: The source booklet states it 'may not be reproduced or translated in whole or in part in any manner without the permission of The International Football Associa ...


## 3. Ingestion method

`scripts/ingest.py` extracts text per page, strips PDF control characters (the
source uses U+0007 as a bullet), removes the leading page-number line and the
running `Laws of the Game 2025/26 | <section>` footer (reused as section
metadata), and packs **whole sentences** into ~1100-char chunks with
whole-sentence overlap — so no chunk starts mid-word. Page numbers are kept as
metadata, not as searchable text.

The PDF is not committed, so we load the prebuilt `chunks.json`. To regenerate:
`python scripts/ingest.py "Laws of the Game 2025_26_single pages.pdf"`.

In [3]:
chunks = json.loads((ROOT / "data" / "chunks.json").read_text())
print("chunks:", len(chunks), "| keys:", list(chunks[0].keys()))
sample = next(c for c in chunks if c["page"] == 103)
print(f"\n[p.{sample['page']}] section={sample['section']!r}")
print(sample["text"][:320])

chunks: 267 | keys: ['id', 'page', 'section', 'text']

[p.103] section='Law 11 — Offside'
1. Offside position It is not an offence to be in an offside position. A player is in an offside position if: • any part of the head, body or feet is in the opponents' half (excluding the halfway line) and • any part of the head, body or feet is nearer to the opponents' goal line than both the ball and the second-last 


## 4. Chunk statistics

In [4]:
import statistics
lens = [len(c["text"]) for c in chunks]
pages = sorted({c["page"] for c in chunks})
ctrl = sum(1 for c in chunks if any(ord(ch) < 9 or 11 <= ord(ch) <= 31 for ch in c["text"]))
print(f"chunks: {len(chunks)}")
print(f"page range: {min(pages)}-{max(pages)}  ({len(pages)} distinct text pages)")
print(f"chunk chars  min/mean/max: {min(lens)} / {statistics.mean(lens):.0f} / {max(lens)}")
print(f"chunks containing control chars: {ctrl}  (should be 0)")
print(f"chunks with a section label: {sum(1 for c in chunks if c.get('section'))}")

chunks: 267
page range: 4-217  (163 distinct text pages)
chunk chars  min/mean/max: 93 / 797 / 1100
chunks containing control chars: 0  (should be 0)
chunks with a section label: 265


## 5. Retrieval (`rag/`)

In [5]:
retriever = Retriever(chunks, DEFAULT_CONFIG)
for r in retriever.search("How long is the half-time interval?"):
    print(f"  rank {r.rank}  p.{r.page:>3}  score {r.score:5.2f}  {r.preview(70)}")

  rank 1  p. 87  score 17.97  1. Periods of play • A match lasts for two equal halves of 45 minutes…
  rank 2  p. 58  score 13.04  5. Offences and sanctions • If a named substitute starts a match inst…
  rank 3  p. 29  score 12.91  • Competitions must decide who will help the referee time the dismiss…
  rank 4  p. 56  score 11.33  Extra time • If a team has not used the maximum number of substitutes…
  rank 5  p. 20  score 10.69  • The universality of the Laws of the Game means that the game is ess…


## 6. Query expansion (token-boundary aware)

Colloquial -> official terminology, matched on **word boundaries** so it never
fires inside a larger word.

In [6]:
for q in ["what happens after a red card", "where does the goalkeeper stand",
          "where does the keeper stand", "every variation of the rule", "what is var"]:
    print(f"{q!r:48} -> fired={matched_phrases(q)}")
print("\nexpanded:", expand_query("what happens after a red card"))

'what happens after a red card'                  -> fired=['red card']
'where does the goalkeeper stand'                -> fired=[]
'where does the keeper stand'                    -> fired=['keeper']
'every variation of the rule'                    -> fired=[]
'what is var'                                    -> fired=['var']

expanded: what happens after a red card sending-off sent-off send-off offences


## 7. Conversation-aware retrieval

A deterministic builder detects referential follow-ups and prepends the most
recent **user** turn's subject (assistant text is never used), keeping retrieval
on topic. This fixes the demonstrated *offside -> "what are the exceptions?"* bug.

In [7]:
history = [{"role":"user","content":"What is the offside rule?"},
           {"role":"assistant","content":"Offside is defined in Law 11 ... [p. 103]."}]
rq = build_retrieval_query("What are the exceptions?", history)
print("rewritten:", rq.rewritten)
print("retrieval query:", rq.text)
print("standalone unchanged:", build_retrieval_query("When is a handball an offence?", history).rewritten)

rewritten: True
retrieval query: What are the exceptions? offside rule
standalone unchanged: False


## 8. Page-diverse retrieval

Keeping the best chunk per page covers up to `top_k` distinct pages instead of
spending context on duplicates.

In [8]:
q = "How long is the half-time interval?"
dd = [r.page for r in retriever.search(q, deduplicate_pages=True)]
nd = [r.page for r in retriever.search(q, deduplicate_pages=False)]
print("page-dedup  top-5 pages:", dd, " unique:", len(set(dd)))
print("chunk-level top-5 pages:", nd, " unique:", len(set(nd)))

page-dedup  top-5 pages: [87, 58, 29, 56, 20]  unique: 5
chunk-level top-5 pages: [87, 58, 29, 56, 20]  unique: 5


## 9. Evaluation methodology

**Metric - Page Hit Rate@k (`hit@k`):** a query is a *hit at k* if any top-k
chunk is on a gold (answer-containing) page. **This is not IR recall.** We also
report page-level MRR@5 and mean unique pages@5, with 95% Wilson CIs.

> The 20 questions are a **development set** (`laws_dev_v1`): they were used to
> develop the expansion map, so these numbers are **optimistic, not held-out**.
> Page labels are coarse and do not establish *answer* correctness.

We display the committed `eval/results.json` (regenerated by
`python eval/evaluate.py`); this cell does not overwrite it.

In [9]:
res = json.loads((ROOT / "eval" / "results.json").read_text())
print("dataset:", res["run"]["dataset"], "| corpus chunks:", res["run"]["corpus"]["chunk_count"])
print(f"{'system':52} {'hit@1':>6}{'hit@3':>6}{'hit@5':>7}{'MRR@5':>7}{'uniq@5':>7}")
for s in res["systems"]:
    m = s["metrics"]
    print(f"{s['name'][:52]:52} {m['hit@1']:6.2f}{m['hit@3']:6.2f}{m['hit@5']:7.2f}"
          f"{m['page_mrr@5']:7.2f}{m['mean_unique_pages@5']:7.2f}")

dataset: {'name': 'laws_dev_v1', 'role': 'development', 'n': 20} | corpus chunks: 267
system                                                hit@1 hit@3  hit@5  MRR@5 uniq@5
BM25 + expansion + page-dedup (production)             0.45  0.60   0.75   0.55   5.00
BM25 + expansion, chunk-level (no dedup)               0.45  0.60   0.75   0.55   4.45
BM25 plain + page-dedup (ablation: no expansion)       0.45  0.60   0.65   0.53   5.00
BM25 plain, chunk-level (no expansion, no dedup)       0.45  0.60   0.65   0.53   4.45
TF-IDF cosine (baseline)                               0.30  0.50   0.55   0.40   4.65
Random (floor)                                         0.05  0.05   0.05   0.05   5.00


In [10]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
names = [s["name"].split(" (")[0] for s in res["systems"]]
hit5 = [s["metrics"]["hit@5"] for s in res["systems"]]
fig, ax = plt.subplots(figsize=(8,3.2))
ax.barh(range(len(names)), hit5, color="#2D6A4F")
ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=8)
ax.invert_yaxis(); ax.set_xlim(0,1); ax.set_xlabel("Page Hit Rate@5 (development set)")
for i,v in enumerate(hit5): ax.text(v+0.01, i, f"{v:.2f}", va="center", fontsize=8)
ax.set_title("Page Hit Rate@5 - development set (n=20)")
plt.tight_layout(); plt.show()

## 10. Per-query error analysis

In [11]:
prod = res["systems"][0]
print("Production misses (gold page never in top-5):")
for m in prod["misses"]:
    print(f"  {m['question'][:60]:60} got {m['retrieved_pages']}  gold {m['gold_pages']}")

plain = next(s for s in res["systems"] if s["name"].startswith("BM25 plain + page"))
pp = {p["question"]: p["hit@5"] for p in plain["per_query"]}
print("\nQuery-level expansion effect:")
for p in prod["per_query"]:
    if p["expansion_phrases"]:
        was, now = pp[p["question"]], p["hit@5"]
        tag = "SAME" if was==now else ("IMPROVED" if now else "REGRESSED")
        print(f"  [{tag:8}] {p['expansion_phrases']}  {p['question'][:50]}")

Production misses (gold page never in top-5):
  Where must the goalkeeper stand during a penalty kick?       got [135, 99, 116, 199, 171]  gold [129]
  How far away must the defensive wall be from the ball at a f got [204, 91, 207, 20, 117]  gold [126, 200]
  How long are the periods of extra time?                      got [56, 29, 87, 20, 58]  gold [97]
  What are the cautionable offences punished by a yellow card? got [31, 30, 114, 28, 18]  gold [115, 116, 183]
  When is a goal scored?                                       got [60, 194, 169, 178, 139]  gold [97]

Query-level expansion effect:
  [SAME    ] ['how long']  How long is the half-time interval?
  [IMPROVED] ['red card']  For which offences is a player sent off with a red
  [IMPROVED] ['how long']  How long does a football match last?
  [SAME    ] ['how long']  How long are the periods of extra time?
  [SAME    ] ['penalty shoot', 'shoot-out']  How many kicks does each team take in a penalty sh
  [SAME    ] ['yellow card']  

## 11. Uncertainty

In [12]:
ci = prod["metrics"]["hit@5_wilson_ci95"]
print(f"Production hit@5 = {prod['metrics']['hit@5']} ({prod['metrics']['hit@5_count']})")
print(f"95% Wilson CI: [{ci[0]}, {ci[1]}]")
print("With n=20 the interval is wide; a CI is NOT evidence of generalization.")

Production hit@5 = 0.75 (15/20)
95% Wilson CI: [0.531, 0.888]
With n=20 the interval is wide; a CI is NOT evidence of generalization.


## 12. Answer-level evaluation framework

Retrieval hit rate does not establish answer correctness. `eval/evaluate_answers.py`
adds a **human-scored** rubric plus deterministic, key-free **citation checks**
(cited pages must have been retrieved). Human scores are pending and not
fabricated.

In [13]:
def citation_check(answer, retrieved_pages):
    cited = extract_citations(answer)
    supported, unsupported = validate_citations(cited, retrieved_pages)
    return {"cited": cited, "supported": supported, "unsupported": unsupported}
print(citation_check("Half-time is 15 minutes [p. 87].", [87, 88]))
print(citation_check("See [p. 9999].", [87, 88]))

{'cited': [87], 'supported': [87], 'unsupported': []}
{'cited': [9999], 'supported': [], 'unsupported': [9999]}


## 13. Optional generation demo

Runs only if `GROQ_API_KEY` is set; otherwise it **skips cleanly**. This means the
committed notebook does not by itself reproduce an LLM answer - run with a key to
see generation. We never print the key.

In [14]:
import urllib.request
key = os.environ.get("GROQ_API_KEY", "").strip()
if not key:
    print("GROQ_API_KEY not set -> skipping generation (retrieval above is fully reproducible).")
else:
    model = os.environ.get("GROQ_MODEL", "llama-3.1-8b-instant")
    rs = retriever.search("How long is the half-time interval?")
    ctx = "\n\n---\n\n".join(f"[Page {r.page}]\n{r.text}" for r in rs)
    sys_prompt = ('Answer ONLY from context as JSON '
                  '{"status":...,"answer":...,"cited_pages":[...]} citing [p. N].')
    body = json.dumps({"model": model, "temperature": 0, "max_tokens": 300,
        "response_format": {"type":"json_object"},
        "messages":[{"role":"system","content":sys_prompt},
            {"role":"user","content":f"Context:\n{ctx}\n\nQuestion: How long is half-time?"}]}).encode()
    req = urllib.request.Request("https://api.groq.com/openai/v1/chat/completions",
        data=body, headers={"Authorization": f"Bearer {key}", "Content-Type":"application/json"})
    out = json.loads(urllib.request.urlopen(req, timeout=30).read())
    print(out["choices"][0]["message"]["content"])

GROQ_API_KEY not set -> skipping generation (retrieval above is fully reproducible).


## 14. Limitations & honest interpretation

- **Development-set leakage:** the 20 questions tuned the expansion map; numbers
  are optimistic, not held-out. No independent test set has been labelled yet.
- **Small sample (n=20):** wide confidence intervals.
- **Self-labelled, coarse page-level metric:** a hit means the right *page* was
  retrieved, not that the chunk contains the answer span, and not that the
  generated answer is correct.
- **Limited answer-level human evaluation:** the harness exists; human scores are
  pending.
- **One document, one language, one edition;** football-specific prompt/expansion
  logic (adaptable to another corpus with domain config, **not** corpus-agnostic).
- **Model/provider dependency:** generation needs Groq; we have not compared model
  sizes, so we make no optimality claim about the 8B model.

**What we can say:** on this development set the production retriever reaches
Page Hit Rate@5 = 0.75 (15/20); query expansion adds +0.10 with no regressions;
page-deduplication improves context efficiency (unique pages 4.45 -> 5.0). We do
**not** claim generalization, hallucination elimination, or that BM25/8B are
universally optimal.